In [1]:
import os
import SimpleITK as sitk
import numpy as np


## this function is used to drop the wrong shape masks in a timepoint
def drop_wrong_shape_masks(masks, images, names):
    ## masks: a list of (D, H, W)

    shape_dict = {}
    for mask in masks:
        if mask.shape not in shape_dict:
            shape_dict[mask.shape] = 1
        else:
            shape_dict[mask.shape] += 1

    final_shape = None
    # for shape, cnt in shape_dict.items():
    #     if cnt > 1:
    #         final_shape = shape

    # final_shape = max(shape_dict, key=shape_dict.get)
    final_shape = find_max_shape(shape_dict)
    if not len(final_shape)==0:
        final_shape = final_shape[0]

    for i in range(len(masks) - 1, -1, -1):
        if masks[i].shape != final_shape:
            images.pop(i)
            masks.pop(i)
            names.pop(i)

    return masks, images, names

def load_array(image_path):
    image = sitk.ReadImage(image_path)
    image = sitk.GetArrayFromImage(image).astype(np.float32)
    return image

def find_max_shape(items):
    max_value = max(items.values())
    max_items = [item for item, quantity in items.items() if quantity == max_value and item[0] > 32]
    if len(max_items) > 1:
        max_items = [max(max_items, key=lambda x: x[1])]
    return max_items

base_dir = '/home/jincan/lrep1/Data/Imaging'
mask_cat_dir = 'DeepBraTumIA-segmentation/native/segmentation/t1_seg_mask.nii.gz'

case_images = []
case_masks =  []
case_names = []
for patient in sorted(os.listdir(base_dir)):
    # print(patient)
    patient_dir = os.path.join(base_dir, patient)

    local_images = []
    local_masks = []
    local_names = []

    tp_list = sorted(os.listdir(patient_dir))
    if len(tp_list) <= 1:
        print(f'NOT enough timepoint: {patient}')
        continue

    for tp in tp_list:

        mod_list = sorted(os.listdir(os.path.join(patient_dir, tp)))
        # if 'T1.nii.gz' not in mod_list:
        #     print(f'NO T1: {patient}/{tp}')
        #     continue
        print(tp)
        tp_dir = os.path.join(patient_dir, tp)

        image_path = os.path.join(tp_dir, 'T1.nii.gz')
        mask_path = os.path.join(tp_dir, mask_cat_dir)
        # print(image_path)
        # print(mask_path)

        ## check if the image and mask exist
        if not os.path.exists(image_path):
            print(f'NO image: {patient}/{tp}')
            continue
            
        if not os.path.exists(mask_path):
            print(f'NO mask: {patient}/{tp}')
            continue

        image = load_array(image_path)
        mask = load_array(mask_path)

        ## check the shape of mask and image
        if image.shape != mask.shape:
            print(f'shape not match: {patient}/{tp}')
            print(f'image shape: {image.shape}')
            print(f'mask shape: {mask.shape}')
            continue

        local_images.append(image)
        local_masks.append(mask)
        local_names.append(f'{patient}_{tp}')
    
    
    local_masks, local_images, local_names = drop_wrong_shape_masks(local_masks, local_images, local_names)
    

    if len(local_images) <= 1:
        print(f'NOT enough timepoint: {patient}')
        continue

    local_images = np.array(local_images)
    local_masks = np.array(local_masks)
    print(local_images.shape)
    print(local_masks.shape)

    case_images.append(local_images)
    case_masks.append(local_masks)
    case_names.append(local_names)
    

print(f'number of cases: {len(case_images)}')



week-000-1
week-000-2
week-044
week-056
NOT enough timepoint: Patient-001
week-000
week-003
week-021
week-037
week-040-1
NO image: Patient-002/week-040-1
week-040-2
week-047
NOT enough timepoint: Patient-002
week-000-1
NO image: Patient-003/week-000-1
week-000-2
week-014
week-027
week-038
NOT enough timepoint: Patient-003
week-000-1
week-000-2
week-020
week-038
week-041
week-057
week-071
week-086
(7, 160, 256, 256)
(7, 160, 256, 256)
week-000-1
week-000-2
week-015
NOT enough timepoint: Patient-005
week-000
week-001
week-015
week-027
week-039
week-053
week-067
week-083
week-097
week-112
week-114
week-121
week-131
week-135
NOT enough timepoint: Patient-006
week-000
week-001
week-015
week-028
week-041
week-055
week-064
week-075
week-089
week-105
NOT enough timepoint: Patient-007
week-000
week-002
week-017
week-066
week-085
NOT enough timepoint: Patient-008
week-000-1
week-000-2
week-014
week-015
week-024
week-026
NO mask: Patient-009/week-026
week-066
NOT enough timepoint: Patient-009
wee

In [33]:
items = {
    (2, 43, 234, 234): 10,
    (3, 22, 150, 234): 15,
    (2, 33, 400, 300): 15,
    (1, 34, 120, 100): 15
}

# 1. 找到字典中所有 value 的最大值
max_value = max(items.values())

# 2. 获取所有 value 等于最大值的数组（key），并且第 2 个值大于 32
max_items = [item for item, quantity in items.items() if quantity == max_value and item[1] > 32]

# 3. 如果 max_items 数量大于 1，按第 2 个值进行进一步比较
if len(max_items) > 1:
    max_items = [max(max_items, key=lambda x: x[1])]

# 4. 打印最终结果
print(f"The item(s) with the highest quantity and second value greater than 32 are: {max_items}")

def find_max_shape(items):
    max_value = max(items.values())
    max_items = [item for item, quantity in items.items() if quantity == max_value and item[1] > 32]
    if len(max_items) > 1:
        max_items = [max(max_items, key=lambda x: x[1])]
    return max_items

The item(s) with the highest quantity and second value greater than 32 are: [(1, 34, 120, 100)]


In [2]:

import h5py

### 4 files for training $ 1 for validation

path_train = 'h5_Data/Imaging/train_long.hdf5'
path_val = 'h5_Data/Imaging/val_long.hdf5'
# path_val = 'example_dataset/cancer/val_placeholder_long.hdf5'

# # 检查目录是否存在，不存在则创建
# if not os.path.exists(path_train):
#     os.makedirs(path_train)
#     print(f"目录 {directory} 已创建。")

# # 检查目录是否存在，不存在则创建
# if not os.path.exists(directory):
#     os.makedirs(directory)
#     print(f"目录 {directory} 已创建。")

## create training .hdf5 file 
with h5py.File(path_train, 'w') as f:
    # for i in range(len(Case_name_list)):
    #     time_points = len(Case_name_list[i])
    #     grp_name = 'subject_' + str(i+1).zfill(3) 
    #     grp = f.create_group(grp_name)
    #     age = np.random.randint(30, 50, size=(time_points, 1))
    #     grp.create_dataset('age', data=age)
    #     grp.create_dataset('t1', data=Case_image_list[i])
    #     grp.create_dataset('mask', data=Case_mask_list[i])

    # cnt = len(Case_name_list)
    cnt = 0
    for i in range(cnt, len(case_names)-1):
        time_points = case_images[i].shape[0]
        grp_name = 'subject_' + str(i).zfill(3) 
        grp = f.create_group(grp_name)
        age = np.random.randint(30, 50, size=(time_points, 1))
        grp.create_dataset('age', data=age)
        grp.create_dataset('image', data=case_images[i])
        grp.create_dataset('mask', data=case_masks[i])


## create validation .hdf5 file 
with h5py.File(path_val, 'w') as f:
    for i in range(len(case_names)):
        len0 = len(case_names)-2
        if i < len0:
            pass
        else:
            time_points = case_images[i].shape[0]
            grp_name = 'subject_' + str(i-len0).zfill(3) 
            grp = f.create_group(grp_name)
            age = np.random.randint(30, 50, size=(time_points, 1))
            grp.create_dataset('age', data=age)
            grp.create_dataset('image', data=case_images[i])
            grp.create_dataset('mask', data=case_masks[i])


In [35]:
cnt = 0

for image in case_images:
    print(image.shape)

    if image.shape[1] > 32 and image.shape[2] > 32 and image.shape[3] > 32:
        cnt += 1

print(cnt)

(2, 24, 640, 640)
(2, 24, 512, 448)
(3, 24, 320, 280)
(7, 160, 256, 256)
(2, 24, 320, 280)
(10, 24, 320, 280)
(9, 24, 320, 280)
(2, 24, 320, 280)
(2, 24, 384, 336)
(3, 24, 320, 280)
(2, 24, 384, 336)
(3, 24, 320, 280)
(3, 160, 256, 256)
(3, 160, 256, 256)
(12, 160, 256, 256)
(4, 24, 320, 280)
(4, 24, 384, 336)
(8, 30, 512, 448)
(2, 24, 384, 336)
(4, 24, 384, 336)
(10, 160, 256, 256)
(3, 24, 384, 324)
(6, 160, 256, 256)
(3, 24, 320, 280)
(5, 160, 256, 256)
(11, 160, 256, 256)
(3, 24, 320, 270)
(5, 24, 320, 280)
(7, 224, 512, 512)
(4, 160, 256, 256)
(4, 160, 256, 256)
(3, 24, 512, 448)
(4, 24, 320, 280)
(3, 160, 256, 256)
(2, 24, 384, 324)
(4, 24, 320, 280)
(12, 160, 256, 256)
(7, 24, 320, 280)
(3, 24, 320, 280)
(2, 24, 320, 280)
(5, 160, 256, 256)
(3, 24, 384, 336)
(2, 24, 320, 280)
(9, 160, 256, 256)
(4, 30, 512, 512)
(3, 24, 384, 336)
(7, 24, 320, 280)
(2, 160, 256, 256)
(2, 24, 384, 336)
(8, 24, 384, 336)
(2, 24, 512, 448)
(3, 25, 384, 384)
(4, 24, 512, 448)
(7, 22, 512, 512)
(3, 24,